In [ ]:
from pathlib import Path
import os
from hashlib import sha256
from safetensors.torch import save_file, load_file
from fileformer.tokenizer import ByteLevelTokenizer
import torch
from torch import Tensor

In [ ]:
META_END_MARKERS = [b'IDAT', b'data', b'mdat', b'\xFF\xDA', b'\x0A\x0A']
PATH_SOUSE_DATA = Path('ex')
CHUNK_SIZE = 2048
tokenizer = ByteLevelTokenizer()
Path("tt").mkdir(exist_ok=True)

In [ ]:
def _tensor_from_tokens(tokens, dtype=torch.float) -> Tensor:
    return torch.tensor(tokens, dtype=dtype)

In [ ]:
def process_file(file_path: Path, output_base: Path):
    """Обрабатывает один файл: отделяет метаданные, сохраняет их и данные по частям."""
    # Создаём выходную директорию для файла
    out_dir = output_base / file_path.name
    out_dir.mkdir(exist_ok=True)

    with open(file_path, 'rb') as f:
        # --- Поиск маркера конца метаданных ---
        non_empty_markers = [m for m in META_END_MARKERS if m]
        meta_buffer = bytearray()
        data_remainder = b''

        if non_empty_markers:
            search_buf = bytearray()
            max_marker_len = max(len(m) for m in non_empty_markers)
            marker_found = False

            while True:
                chunk = f.read(CHUNK_SIZE)
                if not chunk:
                    break
                search_buf.extend(chunk)

                # Поиск первого вхождения любого маркера
                best_pos = None
                best_marker = None
                for marker in non_empty_markers:
                    pos = search_buf.find(marker)
                    if pos != -1:
                        if best_pos is None or pos < best_pos:
                            best_pos = pos
                            best_marker = marker

                if best_pos is not None:
                    # Маркер найден
                    meta_buffer.extend(search_buf[:best_pos])
                    data_remainder = search_buf[best_pos + len(best_marker):]
                    marker_found = True
                    break
                else:
                    # Сохраняем часть буфера, оставляя хвост для перекрытия
                    if len(search_buf) > max_marker_len:
                        save_len = len(search_buf) - max_marker_len
                        meta_buffer.extend(search_buf[:save_len])
                        # Оставляем только хвост, где может начаться маркер
                        search_buf = search_buf[save_len:]

            if not marker_found:
                # Маркер не найден – весь файл считаем метаданными
                meta_buffer.extend(search_buf)
                data_remainder = b''
        else:
            # Нет ни одного непустого маркера – метаданных нет, весь файл — данные
            meta_buffer = bytearray()
            # Файл ещё не читался, указатель в начале

        print(f"File: {file_path.name}, metadata size: {len(meta_buffer)} bytes")

        # Сохраняем метаданные
        if meta_buffer or True:  # всегда сохраняем, даже пустые (можно убрать условие)
            hash_hex = sha256(meta_buffer).hexdigest()
            hash_tokens = _tensor_from_tokens(tokenizer.encode(hash_hex))
            meta_tokens = _tensor_from_tokens(tokenizer.encode(meta_buffer.hex()))
            save_file(
                {'hash_tokens': hash_tokens, 'tokenized_metadata': meta_tokens},
                out_dir / 'meta.safetensors'
            )

        # --- Обработка данных чанками ---
        chunk_number = 0
        current_chunk = bytearray(data_remainder)  # остаток от буфера поиска

        while True:
            # Добираем данные до полного чанка (CHUNK_SIZE)
            while len(current_chunk) < CHUNK_SIZE:
                more = f.read(CHUNK_SIZE - len(current_chunk))
                if not more:
                    break
                current_chunk.extend(more)

            if current_chunk:
                # Вычисляем хеш и токенизируем данные чанка
                hash_hex = sha256(current_chunk).hexdigest()
                hash_tokens = _tensor_from_tokens(tokenizer.encode(hash_hex))
                data_tokens = _tensor_from_tokens(tokenizer.encode(current_chunk.hex()))

                save_file(
                    {'hash_tokens': hash_tokens, 'tokenized_data': data_tokens},
                    out_dir / f'data{chunk_number}.safetensors'
                )

                chunk_number += 1
                current_chunk = bytearray()  # готовим для следующего чанка
            else:
                break

        print(f"Total chunks: {chunk_number}\n")

In [ ]:
output_root = Path("tt")
output_root.mkdir(exist_ok=True)

for file in PATH_SOUSE_DATA.iterdir():
    if not file.is_file():
        continue
    process_file(file, output_root)

In [ ]:
files = [x for x in Path('/Users/daniilogorodnikov/PycharmProjects/Notus/fileformer/file_dataset/test_out').glob('*/**') if x.is_file() and x.stat().st_size > 0]

In [ ]:
folder_names = [item.name for item in Path('/Users/daniilogorodnikov/PycharmProjects/Notus/fileformer/file_dataset/test_out').iterdir() if item.is_dir()]

In [ ]:
len(files) - len(folder_names)

In [ ]:
files

In [ ]:
folder_names in files

In [ ]:
prob_tensor = torch.full((5,5), 0.3)
mask = torch.bernoulli(prob_tensor).to(torch.int8)

In [ ]:
mask.shape

In [ ]:
torch.randint(1, 565, mask.shape)*(1 - mask)

In [6]:
from fileformer import FileDataset
from torch.utils.data import DataLoader

In [ ]:
dataset = FileDataset("/Users/daniilogorodnikov/PycharmProjects/Notus/fileformer/file_dataset/test_out", 0.3)

In [ ]:
loader = DataLoader(dataset, batch_size=2)

In [ ]:
x = next(iter(loader))

In [ ]:
x[1][0]

In [20]:
from pathlib import Path
import os
from hashlib import sha256
from safetensors.torch import save_file, load_file
from fileformer.tokenizer import ByteLevelTokenizer
import torch
from torch import Tensor

In [23]:
from fileformer import ENWIK8Dataset

In [48]:
dataset = ENWIK8Dataset('97957-9.wav', ByteLevelTokenizer(), 8192, 0, "cache")

In [49]:
loader = DataLoader(dataset, batch_size=33, shuffle=False)

In [50]:
x = next(iter(loader))

In [51]:
unique, counts = torch.unique(x, return_counts=True)

In [52]:
probs = counts.float() / counts.sum()

In [53]:
entropy = -torch.sum(probs * torch.log(probs))

In [54]:
entropy

tensor(5.1731)

In [56]:
-torch.log(probs)

tensor([3.8631, 5.7017, 5.7039, 5.7298, 5.7140, 5.7389, 5.6531, 5.7355, 5.7575,
        5.7230, 5.7788, 5.7061, 5.7705, 5.6787, 5.7528, 5.8584, 5.7435, 5.7740,
        5.7884, 5.7752, 5.7447, 5.7598, 5.7681, 5.7740, 5.7563, 5.7598, 5.7552,
        5.7872, 5.7933, 5.7981, 5.7776, 5.7716, 5.8341, 5.8392, 5.7681, 5.8129,
        5.8067, 5.8316, 5.8533, 5.7622, 5.8456, 5.8649, 5.8092, 5.8392, 5.8781,
        5.8055, 5.8379, 5.8715, 5.8546, 5.8153, 5.8481, 5.9008, 5.8794, 5.8927,
        5.8468, 5.8715, 5.7920, 5.4872, 4.9110, 3.8955, 2.9414, 3.3827, 4.2592,
        4.6577, 5.6281, 5.8571, 5.8241, 5.8636, 5.8820, 5.8675, 5.9171, 5.8767,
        5.8900, 5.8291, 5.8702, 5.9048, 5.9021, 5.8954, 5.8873, 5.8994, 5.8636,
        5.9035, 5.8443, 5.9310, 5.9116, 5.9062, 5.9008, 5.8887, 5.8927, 5.9897,
        5.9185, 5.8533, 5.9089, 5.8927, 5.9035, 5.9130, 5.8834, 5.9008, 5.9212,
        5.9578, 5.8927, 5.8860, 5.9130, 5.9678, 5.8741, 5.9365, 5.9736, 5.9048,
        5.8981, 5.9076, 5.8967, 5.9048, 